# NB6.0 — Extract NAS-Bench-201 to a small CSV
____________________________________________________________________________________________________

NAS-Bench-201 (Dong & Yang, 2020) trained **all 15,625 architectures** of its search space on CIFAR-10, CIFAR-100 and ImageNet16-120 and published every result in one 2.2 GB file (`NAS-Bench-201-v1_0-e61699.pth`).

This notebook reads that file once and keeps only what we need for the reproduction in NB6.1: for each architecture its **cell string** and its **accuracies after 200 epochs** (averaged over the training seeds, as the official API does):

| column | NAS-Bench-201 dataset | metric |
|---|---|---|
| `cifar10_valid` | `cifar10-valid` (trained on 25k, evaluated on the other 25k training images) | `x-valid@199` |
| `cifar10_test` | `cifar10` (trained on the 50k training images) | `ori-test@199` |
| `cifar100_valid` / `cifar100_test` | `cifar100` | `x-valid@199` / `x-test@199` |
| `imagenet16_valid` / `imagenet16_test` | `ImageNet16-120` | `x-valid@199` / `x-test@199` |

These are the same metrics as in Table 2 of the NASWOT paper (Mellor et al., 2021).


In [1]:
import os, time
import numpy as np
import pandas as pd
import torch

def find_path(name, search_roots):
    for search_root in search_roots:
        for root, dirs, files in os.walk(search_root):
            if name in dirs or name in files:
                return os.path.join(root, name)
    return None

BENCH_FILE = find_path("NAS-Bench-201-v1_0-e61699.pth", ["/kaggle/input", "..", "../.."])
print(BENCH_FILE)
start = time.time()
bench = torch.load(BENCH_FILE, map_location="cpu", weights_only=False)   # a big python dict (pickle)
print("loaded in %.0f s" % (time.time() - start))
print(type(bench), list(bench.keys()))

/kaggle/input/datasets/shifatearman/nasbench201/NAS-Bench-201-v1_0-e61699.pth
loaded in 36 s
<class 'dict'> ['meta_archs', 'total_archs', 'arch2infos', 'evaluated_indexes']


**Structure of the file** (we print one entry to know where the accuracies are).

In [2]:
print("number of architectures:", len(bench["meta_archs"]))
print("arch 0:", bench["meta_archs"][0])
info0 = bench["arch2infos"][0]
print(type(info0), list(info0.keys()) if isinstance(info0, dict) else info0)
full0 = info0["full"]
print(list(full0.keys()))
print("datasets / seeds:", full0["dataset_seed"])
key0 = list(full0["all_results"].keys())[0]
res0 = full0["all_results"][key0]
print(key0, list(res0.keys()))
print("eval_acc1es keys:", [k for k in res0["eval_acc1es"].keys() if k.endswith("@199")])

number of architectures: 15625
arch 0: |avg_pool_3x3~0|+|nor_conv_1x1~0|skip_connect~1|+|nor_conv_1x1~0|skip_connect~1|skip_connect~2|
<class 'dict'> ['full', 'less']
['arch_index', 'arch_str', 'all_results', 'dataset_seed', 'clear_net_done']
datasets / seeds: {'cifar10-valid': [777, 888, 999], 'cifar10': [777, 888, 999], 'cifar100': [777, 888, 999], 'ImageNet16-120': [777, 888, 999]}
('cifar10-valid', 777) ['name', 'net_state_dict', 'train_acc1es', 'train_acc5es', 'train_losses', 'train_times', 'arch_config', 'params', 'flop', 'seed', 'epochs', 'latency', 'eval_names', 'eval_acc1es', 'eval_times', 'eval_losses']
eval_acc1es keys: ['x-valid@199', 'ori-test@199']


In [3]:
def mean_acc(full, dataset, metric):
    # average over the seeds, like the official API (is_random=False)
    seeds = full["dataset_seed"][dataset]
    return float(np.mean([full["all_results"][(dataset, seed)]["eval_acc1es"][metric] for seed in seeds]))

rows = []
for idx, arch_str in enumerate(bench["meta_archs"]):
    full = bench["arch2infos"][idx]["full"]
    rows.append({
        "index": idx, "arch": arch_str,
        "cifar10_valid": mean_acc(full, "cifar10-valid", "x-valid@199"),
        "cifar10_test": mean_acc(full, "cifar10", "ori-test@199"),
        "cifar100_valid": mean_acc(full, "cifar100", "x-valid@199"),
        "cifar100_test": mean_acc(full, "cifar100", "x-test@199"),
        "imagenet16_valid": mean_acc(full, "ImageNet16-120", "x-valid@199"),
        "imagenet16_test": mean_acc(full, "ImageNet16-120", "x-test@199"),
    })
df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/nasbench201_accuracies.csv" if os.path.exists("/kaggle/working") else "nasbench201_accuracies.csv", index=False)
print(df.head())
print(df.describe())

   index                                               arch  cifar10_valid  \
0      0  |avg_pool_3x3~0|+|nor_conv_1x1~0|skip_connect~...      81.982667   
1      1  |nor_conv_3x3~0|+|nor_conv_3x3~0|avg_pool_3x3~...      90.769333   
2      2  |avg_pool_3x3~0|+|nor_conv_3x3~0|nor_conv_3x3~...      81.910667   
3      3  |avg_pool_3x3~0|+|skip_connect~0|none~1|+|none...      84.377333   
4      4  |skip_connect~0|+|skip_connect~0|nor_conv_1x1~...      85.125333   

   cifar10_test  cifar100_valid  cifar100_test  imagenet16_valid  \
0     85.863333       52.700000      52.913333         28.211111   
1     93.660000       69.826667      70.293333         44.488889   
2     85.740000       54.300000      54.866667         27.633333   
3     86.616667       57.413333      58.306667         29.433333   
4     88.216667       59.566667      60.140000         32.144444   

   imagenet16_test  
0        26.633333  
1        44.033333  
2        27.255555  
3        28.044444  
4        31.48888

**Sanity check** against the numbers of the NASWOT paper (Table 2): *Random* = mean ± std over the whole benchmark, CIFAR-10 test 86.61 ± 13.46, CIFAR-100 test 60.83 ± 12.58, ImageNet16-120 test 33.13 ± 9.66; the best architecture reaches 94.37 % on CIFAR-10 test.

In [4]:
for col in df.columns[2:]:
    print("%-17s mean %.2f  std %.2f  max %.2f" % (col, df[col].mean(), df[col].std(), df[col].max()))

cifar10_valid     mean 83.65  std 12.82  max 91.61
cifar10_test      mean 87.04  std 12.94  max 94.37
cifar100_valid    mean 61.25  std 12.14  max 73.49
cifar100_test     mean 61.36  std 12.17  max 73.51
imagenet16_valid  mean 33.77  std 9.25  max 46.77
imagenet16_test   mean 33.54  std 9.52  max 47.31
